# 00 Project Setup

This notebook checks that the project environment is ready for the Knee OA KL grading project.

Main goals:

1. Check Python and package versions.
2. Check whether PyTorch can see the GPU.
3. Set random seeds for reproducibility.
4. Define the project folder structure.
5. Create folders needed for data, models, logs, and results.
6. Document the Kaggle dataset source.
7. Create a GitHub-safe `.gitignore` file.

This notebook should be run before training or evaluation.

In [ ]:
# This cell imports basic Python libraries used for project setup.
# These libraries help with file paths, system information, randomness, and saving setup reports.

from pathlib import Path
import os
import sys
import random
import platform
import json
from datetime import datetime

In [ ]:
# This cell defines the main project folder.
# It is written so the notebook works whether you run it from:
# 1. the project root folder
# 2. the notebooks/ folder

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

print("Current working directory:", current_dir)
print("Project root:", PROJECT_ROOT)

In [ ]:
# This cell creates the standard folder structure for the project.
# These folders make the work easier to reproduce on GitHub and HPC.

folders = [
    "configs",
    "notebooks",
    "src/knee_oa",
    "scripts",
    "slurm",
    "data/raw",
    "data/processed",
    "data/splits",
    "outputs/checkpoints",
    "outputs/predictions",
    "outputs/figures",
    "outputs/logs",
    "outputs/reports",
]

for folder in folders:
    folder_path = PROJECT_ROOT / folder
    folder_path.mkdir(parents=True, exist_ok=True)

print("Project folders created or already exist.")

In [ ]:
# This cell checks the Python version and basic system information.
# This is useful for reproducibility because another person can compare their setup with yours.

python_info = {
    "python_version": sys.version,
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "machine": platform.machine(),
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

for key, value in python_info.items():
    print(f"{key}: {value}")

In [ ]:
# This cell checks whether PyTorch is installed.
# It also checks whether CUDA and GPU access are available.
# On the HPC GPU node, CUDA should usually be available.

try:
    import torch

    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("CUDA version used by PyTorch:", torch.version.cuda)
        print("Number of GPUs:", torch.cuda.device_count())

        for gpu_id in range(torch.cuda.device_count()):
            print(f"GPU {gpu_id}:", torch.cuda.get_device_name(gpu_id))
    else:
        print("No GPU detected by PyTorch.")
        print("This is okay on a login node, but training should run on a GPU node.")

except ImportError:
    print("PyTorch is not installed in this environment.")
    print("We need to install PyTorch before training.")

In [ ]:
# This cell sets random seeds.
# Random seeds help make experiments more reproducible.
# They do not guarantee perfect reproducibility on every GPU, but they reduce randomness.

SEED = 42

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

try:
    import numpy as np
    np.random.seed(SEED)
    print("NumPy seed set.")
except ImportError:
    print("NumPy is not installed.")

try:
    import torch
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)

    # These settings improve reproducibility.
    # They may make training slightly slower.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print("PyTorch seed set.")
except ImportError:
    print("PyTorch is not installed.")

print("Seed:", SEED)

In [ ]:
# This cell saves a small environment report.
# This report helps document exactly where and when the setup notebook was run.

report = {
    "project_root": str(PROJECT_ROOT),
    "python_info": python_info,
    "seed": SEED,
}

try:
    import torch

    report["pytorch"] = {
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "gpu_count": torch.cuda.device_count(),
        "gpu_names": [
            torch.cuda.get_device_name(i)
            for i in range(torch.cuda.device_count())
        ] if torch.cuda.is_available() else [],
    }

except ImportError:
    report["pytorch"] = "PyTorch not installed"

report_path = PROJECT_ROOT / "outputs" / "reports" / "environment_report.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=4)

print("Environment report saved to:")
print(report_path)

## Dataset Source

This project uses the Kaggle Knee Osteoarthritis Dataset with KL grading.

Dataset source:

- Kaggle dataset slug: `tommyngx/kneeoa`
- Dataset page: https://www.kaggle.com/datasets/tommyngx/kneeoa

Raw image data should not be committed to GitHub.

For reproducibility, the dataset should be downloaded using the Kaggle API or stored in a documented HPC data directory.

In [ ]:
# This cell stores the dataset source information.
# We save this information so that another person knows exactly which dataset was used.

DATASET_INFO = {
    "dataset_name": "Knee Osteoarthritis Dataset with KL Grading",
    "source": "Kaggle",
    "kaggle_slug": "tommyngx/kneeoa",
    "dataset_url": "https://www.kaggle.com/datasets/tommyngx/kneeoa",
    "notes": "Raw images should not be committed to GitHub.",
}

DATASET_INFO

In [ ]:
# This cell checks whether the Kaggle command-line tool is installed.
# The Kaggle API is useful for reproducibly downloading the dataset.
# On HPC, internet access may not be available on compute nodes, so you may need to download the dataset elsewhere first.

import subprocess

try:
    result = subprocess.run(
        ["kaggle", "--version"],
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode == 0:
        print("Kaggle API is installed.")
        print(result.stdout.strip())
    else:
        print("Kaggle command found, but it returned an error.")
        print(result.stderr.strip())

except FileNotFoundError:
    print("Kaggle API is not installed in this environment.")
    print("You can install it later with: pip install kaggle")

In [ ]:
# This cell defines where the dataset should live inside the project.
# For GitHub reproducibility, we keep the folder structure but do not commit the raw images.

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
KAGGLE_DATA_DIR = RAW_DATA_DIR / "kaggle_kneeoa"

print("Raw data directory:", RAW_DATA_DIR)
print("Kaggle dataset directory:", KAGGLE_DATA_DIR)

In [ ]:
# This cell gives the command needed to download the dataset from Kaggle.
# It does not run automatically unless RUN_KAGGLE_DOWNLOAD is set to True.
#
# Important:
# - You need a Kaggle account.
# - You need kaggle.json configured.
# - Do not commit kaggle.json to GitHub.
# - On HPC, it may be better to download locally and transfer the data.

RUN_KAGGLE_DOWNLOAD = False

if RUN_KAGGLE_DOWNLOAD:
    KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

    download_command = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        DATASET_INFO["kaggle_slug"],
        "-p",
        str(KAGGLE_DATA_DIR),
        "--unzip",
    ]

    result = subprocess.run(
        download_command,
        capture_output=True,
        text=True,
        check=False,
    )

    print(result.stdout)
    print(result.stderr)

    if result.returncode == 0:
        print("Dataset downloaded successfully.")
    else:
        print("Dataset download failed.")
else:
    print("Kaggle download is currently disabled.")
    print("To download, set RUN_KAGGLE_DOWNLOAD = True after configuring the Kaggle API.")

In [ ]:
# This cell saves the dataset source information to a JSON file.
# This helps make the project reproducible and documents the original data source.

dataset_report_path = PROJECT_ROOT / "outputs" / "reports" / "dataset_source.json"

with open(dataset_report_path, "w") as f:
    json.dump(DATASET_INFO, f, indent=4)

print("Dataset source report saved to:")
print(dataset_report_path)

In [ ]:
# This cell creates a .gitignore file.
# The goal is to prevent raw data, model checkpoints, logs, and private files from being uploaded to GitHub.

gitignore_path = PROJECT_ROOT / ".gitignore"

gitignore_text = """
# Python
__pycache__/
*.pyc
.ipynb_checkpoints/

# Environment files
.env
.venv/
venv/
kaggle.json

# Data
data/raw/
data/processed/

# Outputs
outputs/checkpoints/
outputs/logs/
*.pt
*.pth
*.ckpt

# OS files
.DS_Store
Thumbs.db
"""

with open(gitignore_path, "w") as f:
    f.write(gitignore_text.strip() + "\n")

print(".gitignore created or updated at:")
print(gitignore_path)